# 02 — Exploratory data analysis

Every chart below answers a **question**. Decorative plots are omitted on purpose.

## Questions

1. Are fraudulent amounts clustered at extremes?
2. How severe is class imbalance in **counts** vs **percentages**?
3. Does the two-day `Time` clock show periods with a higher fraud *rate*?
4. Which PCA features even *look* separable by class? (association, not causation)


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_processing import dataset_overview, drop_duplicates_if_present, load_raw_dataset
from src.utils import RAW_DATASET_PATH

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from src.feature_engineering import add_engineered_features

sns.set_theme(style="whitegrid", context="notebook")
df = add_engineered_features(drop_duplicates_if_present(load_raw_dataset()))
print(df.shape)


## Class imbalance

**Question:** how many fraud cases do we actually have to learn from?

**Why:** tree ensembles can still work with hundreds of positives, but resampling and class weights only make sense once we have seen the counts.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = df["Class"].map({0: "Legitimate", 1: "Fraud"})
labels.value_counts().plot(kind="bar", ax=axes[0], color=["#2c5364", "#c45c26"])
axes[0].set_title("Absolute counts")
axes[0].set_ylabel("Transactions")
(labels.value_counts(normalize=True) * 100).plot(kind="bar", ax=axes[1], color=["#2c5364", "#c45c26"])
axes[1].set_title("Percent of dataset")
axes[1].set_ylabel("%")
fig.tight_layout()


## Amount distribution

**Question:** are fraudulent transactions concentrated around particular amounts?

**Why:** a rule like "flag Amount > 1000" is tempting and often wrong. Fraudsters frequently test small charges.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.kdeplot(data=df, x="Amount", hue="Class", common_norm=False, clip=(0, 500), ax=axes[0])
axes[0].set_title("Amount KDE (clipped at 500)")
sns.boxplot(data=df, x="Class", y="Amount", showfliers=False, ax=axes[1])
axes[1].set_title("Amount by class (outliers hidden)")
fig.tight_layout()
df.groupby("Class")["Amount"].describe()


## Time / hour-of-day proxy

**Question:** does fraud rate vary over the 24-hour wrap of `Time`?

`Time` is seconds since the **first transaction in the file**, not a calendar datetime. `(Time % 86400) / 3600` is a convenient daily clock, not "14:00 in Brussels".


In [ ]:
hourly = df.groupby(df["hour_of_day"].astype(int)).agg(
    n=("Class", "size"),
    fraud_rate=("Class", "mean"),
    mean_amount=("Amount", "mean"),
)
fig, ax = plt.subplots(figsize=(8, 4))
hourly["fraud_rate"].plot(ax=ax, color="#c45c26")
ax.set_xlabel("Hour-of-day proxy")
ax.set_ylabel("Fraud rate")
ax.set_title("Fraud rate by hour-of-day proxy")
hourly.head()


## Feature analysis (PCA components)

**Question:** which `V*` distributions differ by class enough to be useful to a model?

We plot a few components that literature and prior runs often highlight (V4, V10, V12, V14, V17). **Different densities ≠ the feature caused fraud.** They are rotated mixtures of hidden raw fields.


In [ ]:
cols = ["V4", "V10", "V12", "V14", "V17"]
fig, axes = plt.subplots(1, 5, figsize=(14, 3), sharey=False)
for ax, col in zip(axes, cols):
    sns.kdeplot(data=df, x=col, hue="Class", common_norm=False, ax=ax, legend=False)
    ax.set_title(col)
fig.tight_layout()


## Correlation snapshot

**Question:** are Amount, hour, and a handful of PCA features linearly associated with `Class`?

Pearson correlation with a 0.17% minority class is a blunt instrument. Weak correlation does **not** mean a nonlinear model cannot use the feature. Strong correlation still is **not causation**.


In [ ]:
corr_cols = cols + ["Amount", "hour_of_day", "Class"]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Pearson correlation (selected columns)")
fig.tight_layout()


## Scatter (sampled)

**Question:** do V14 vs V12 show a visible fraud cloud, or is overlap huge?

Huge overlap is expected: that is why we need a multivariate model and why false positives will exist.


In [ ]:
sample = pd.concat([
    df[df.Class == 0].sample(4000, random_state=42),
    df[df.Class == 1],
], ignore_index=True)
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=sample, x="V12", y="V14", hue="Class", alpha=0.35, ax=ax, palette={0: "#2c5364", 1: "#c45c26"})
ax.set_title("V12 vs V14 (all fraud + 4000 legit)")
